# Module 1: Foundations of Python and Data Handling


## Section 4: Data Cleaning and Preparation

### Objective

- Understand why data cleaning is critical for accurate business analysis
- Identify and handle missing values appropriately
- Detect and remove duplicate records
- Transform data types and standardize formats
- Identify and manage outliers that distort analysis
- Create a complete data cleaning workflow for business datasets

### Main Contents with Examples

```mermaid
mindmap
  root((Data Cleaning<br/>Preparation))
    Business Reality
      80/20 Rule
    Missing Data
      MCAR, MAR, MNAR
      Strategies
    Duplicates
      Detection
      Removal
    Data Types
      Numeric/Datetime
    String Cleaning
      Standardization
    Outliers
      IQR Method
    Pipeline
      Reusable Logic
```


**The Reality of Business Data**

**Why Data is Never Perfect:**

In textbooks and tutorials, data is clean and ready to analyze. Real business data is messy:

- Sales rep leaves "Region" blank on forms
- Customer enters phone number as "555-1234" one time and "(555) 1234" another
- Database crashes mid-entry leaving incomplete records
- Someone accidentally enters sales as "$1,200,000" instead of "$120,000"
- Same customer appears twice with slight name variations

**The 80/20 Rule in Data Analysis:**

Data professionals spend approximately 80% of their time cleaning data and only 20% on actual analysis. This isn't inefficiency - it's necessity. **Garbage in, garbage out** - analysis of dirty data produces unreliable results.

**Types of Data Quality Issues:**

1. **Missing values** - empty cells, NaN (Not a Number), null values
2. **Duplicates** - same record entered multiple times
3. **Inconsistent formats** - "New York" vs "NY" vs "new york"
4. **Wrong data types** - numbers stored as text
5. **Outliers** - extreme values that may be errors or real anomalies
6. **Invalid values** - negative quantities, future dates for past transactions

**Understanding Missing Data**

**Types of Missing Data:**

Not all missing data is the same. Understanding WHY data is missing affects how you handle it:

**1. Missing Completely at Random (MCAR)**

- Pure chance - random data entry errors
- Example: Scanner malfunction skips random records
- Safe to delete these records

**2. Missing at Random (MAR)**

- Missing relates to other variables
- Example: Older customers more likely to skip "email" field
- Consider filling with appropriate values

**3. Missing Not at Random (MNAR)**

- Missingness is meaningful
- Example: High earners don't report income
- Deleting these records creates bias

**Detecting Missing Values in Pandas:**


In [29]:
import pandas as pd
import numpy as np

# Create sample data with missing values
data = {
    "Date": ["2024-01-15", "2024-01-16", None, "2024-01-18"],
    "Product": ["Laptop", "Mouse", "Laptop", "Keyboard"],
    "Sales": [1200, 25, 1200, None],
    "Region": ["North", "South", "North", "South"]
}

df = pd.DataFrame(data)

# Check for missing values
print(df.isnull())           # True/False for each cell
print(df.isnull().sum())     # Count per column
print(df.info())             # Shows non-null counts


    Date  Product  Sales  Region
0  False    False  False   False
1  False    False  False   False
2   True    False  False   False
3  False    False   True   False
Date       1
Product    0
Sales      1
Region     0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Date     3 non-null      object 
 1   Product  4 non-null      object 
 2   Sales    3 non-null      float64
 3   Region   4 non-null      object 
dtypes: float64(1), object(3)
memory usage: 260.0+ bytes
None


**Output:**
```
    Date  Product  Sales  Region
0  False    False  False   False
1  False    False  False   False
2   True    False  False   False
3  False    False   True   False
Date       1
Product    0
Sales      1
Region     0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Date     3 non-null      object 
 1   Product  4 non-null      object 
 2   Sales    3 non-null      float64
 3   Region   4 non-null      object 
dtypes: float64(1), object(3)
memory usage: 260.0+ bytes
None
```

**Strategies for Handling Missing Values:**

**Strategy 1: Deletion (when data is MCAR and abundant)**


In [30]:
# Remove rows with ANY missing values
df_clean = df.dropna()

# Remove rows where specific column is missing
df_clean = df.dropna(subset=["Sales"])


**When to use:** Dataset is large, few missing values, missingness is random.

**Strategy 2: Fill with Statistical Measure**


In [31]:
# Fill missing sales with average
mean_sales = df["Sales"].mean()
df["Sales"] = df["Sales"].fillna(mean_sales)

# Or median (better for skewed data)
median_sales = df["Sales"].median()
df["Sales"] = df["Sales"].fillna(median_sales)


**When to use:** Numeric data, missingness is random, want to preserve dataset size.

**Strategy 3: Fill with Placeholder**


In [32]:
# Fill missing region with "Unknown"
df["Region"] = df["Region"].fillna("Unknown")

# Fill missing dates with specific value
df["Date"] = df["Date"].fillna("2024-01-17")


**When to use:** Categorical data, want to flag missing as separate category.

**Strategy 4: Forward/Backward Fill**


In [33]:
# Copy previous value down
df["Region"] = df["Region"].ffill()

# Copy next value up
df["Region"] = df["Region"].bfill()


**When to use:** Time series data where values change slowly.

**Business Decision:** The method you choose should be documented and justified. Different business contexts require different approaches.

**Handling Duplicate Records**

**Why Duplicates Occur:**

- Customer submits order twice (technical issue)
- Data merged from multiple sources
- Database synchronization errors
- Human error in data entry

**Detecting Duplicates:**


In [34]:
# Check for completely duplicate rows
print(df.duplicated())              # True/False for each row
print(df.duplicated().sum())        # Count of duplicates
print(df[df.duplicated()])          # View duplicate rows

# Check for duplicates in specific column
print(df.duplicated(subset=["Product"]))


0    False
1    False
2    False
3    False
dtype: bool
0
Empty DataFrame
Columns: [Date, Product, Sales, Region]
Index: []
0    False
1    False
2     True
3    False
dtype: bool


**Output:**
```
0    False
1    False
2    False
3    False
dtype: bool
0
Empty DataFrame
Columns: [Date, Product, Sales, Region]
Index: []
```

**Removing Duplicates:**


In [35]:
# Remove duplicate rows (keeps first occurrence)
df_unique = df.drop_duplicates()

# Remove duplicates based on specific column
df_unique = df.drop_duplicates(subset=["Product"], keep="first")

# Options for 'keep':
# - "first": Keep first occurrence (default)
# - "last": Keep last occurrence
# - False: Remove all duplicates including original


**Business Consideration:** Before removing duplicates, verify they ARE duplicates:

- Same customer might legitimately make multiple purchases
- Same product might be sold multiple times
- Check all relevant columns, not just one

**Data Type Conversion**

**Why Data Types Matter:**

Pandas must know data types to perform operations:

- Mathematical operations require numeric types
- Date operations require datetime type
- String operations require object/string type

**Common Type Issues:**

- Numbers imported as text: "1500" instead of 1500
- Dates imported as text: "2024-01-15" instead of datetime
- Leading zeros lost: "00123" becomes 123

**Checking and Converting Types:**


In [36]:
# Check current data types
print(df.dtypes)

# Convert string to numeric
df["Sales"] = pd.to_numeric(df["Sales"])

# Convert to integer (after filling NaN first)
df["Sales"] = df["Sales"].fillna(0).astype(int)

# Convert to datetime
df["Date"] = pd.to_datetime(df["Date"])

# Convert with error handling
df["Sales"] = pd.to_numeric(df["Sales"], errors="coerce")
# 'coerce' turns non-numeric values to NaN instead of error


Date        object
Product     object
Sales      float64
Region      object
dtype: object


**Output:**
```
Date        object
Product     object
Sales      float64
Region      object
dtype: object
```

**Benefits of Correct Types:**

- Enable proper sorting (1, 2, 10 instead of 1, 10, 2)
- Allow mathematical operations
- Enable date-based filtering and analysis
- Reduce memory usage

**String Cleaning and Standardization**

**Common String Issues in Business Data:**

- Inconsistent capitalization: "California" vs "california" vs "CALIFORNIA"
- Extra whitespace: " Sales " instead of "Sales"
- Different abbreviations: "New York" vs "NY"
- Special characters: "Department–Sales" vs "Department-Sales"

**Pandas String Methods:**


In [37]:
# Remove leading/trailing whitespace
df["Product"] = df["Product"].str.strip()

# Convert to consistent case
df["Region"] = df["Region"].str.lower()      # all lowercase
df["Region"] = df["Region"].str.upper()      # all uppercase
df["Region"] = df["Region"].str.title()      # Title Case
df["Region"] = df["Region"].str.capitalize() # Capitalize first

# Replace text
df["Product"] = df["Product"].str.replace("Laptop", "Laptop Pro")

# Remove/clean characters (example: standardise region codes)
df["Region"] = df["Region"].str.replace(" ", "_")           # Replace spaces with underscore
df["Region"] = df["Region"].str.replace(r"[^A-Za-z_]", "", regex=True)  # Keep only letters


**Creating Standardized Categories:**


In [38]:
# Map variations to standard values
region_mapping = {
    "north": "North",
    "NORTH": "North",
    "N": "North",
    "Northern": "North"
}
df["Region"] = df["Region"].map(region_mapping)


**Identifying and Handling Outliers**

**What Are Outliers?**

Outliers are data points significantly different from others:
- Sales of $1,000,000 when typical sales are $50,000
- Customer age of 150 years
- Order quantity of -50

**Why They Matter:**

- Could be errors (data entry mistake)
- Could be real anomalies (legitimate big sale, VIP customer)
- Distort averages and statistical analyses
- Can mislead visualizations

**Statistical Method: Interquartile Range (IQR)**

The IQR method is a robust way to identify outliers:


In [39]:
# Calculate quartiles
Q1 = df["Sales"].quantile(0.25)    # 25th percentile
Q3 = df["Sales"].quantile(0.75)    # 75th percentile
IQR = Q3 - Q1                       # Interquartile range

# Define outlier boundaries
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Normal range: ${lower_bound:,.0f} to ${upper_bound:,.0f}")


Normal range: $-269 to $2,082


**Output:**
```
Normal range: $-269 to $2,081
```

**What these numbers mean:**

- **Q1**: 25% of data is below this value
- **Q3**: 75% of data is below this value
- **IQR**: The middle 50% of data falls in this range
- **1.5 × IQR rule**: Standard statistical threshold for outliers

**Handling Outliers:**


In [40]:
# Identify outliers
outliers = df[(df["Sales"] < lower_bound) | (df["Sales"] > upper_bound)]
print("Outliers found:")
print(outliers)

# Option 1: Remove outliers
df_clean = df[(df["Sales"] >= lower_bound) & (df["Sales"] <= upper_bound)]

# Option 2: Cap outliers (replace with boundary values)
df["Sales_Capped"] = df["Sales"].clip(lower=lower_bound, upper=upper_bound)

# Option 3: Replace with median
median_sales = df["Sales"].median()
df["Sales_Clean"] = df["Sales"].apply(
    lambda x: median_sales if (x < lower_bound or x > upper_bound) else x
)


Outliers found:
Empty DataFrame
Columns: [Date, Product, Sales, Region]
Index: []


**Output:**
```
Outliers found:
Empty DataFrame
Columns: [Date, Product, Sales, Region]
Index: []
```

**Business Decision:** Don't automatically remove outliers:

1. Investigate: Is this a data error or legitimate?
2. Consider context: Holiday sales spike might be real
3. Document: Explain why outliers were kept or removed
4. Sometimes: Keep outliers, use median instead of mean

**Creating a Complete Cleaning Pipeline**

**The Professional Approach:**

Rather than cleaning ad-hoc, create a reusable function that applies all cleaning steps consistently:


In [41]:
def clean_sales_data(df):
    """
    Comprehensive data cleaning pipeline for sales data
    
    Steps:
    1. Remove duplicate records
    2. Handle missing values
    3. Standardize text fields
    4. Convert data types
    5. Remove outliers
    
    Returns: Cleaned DataFrame
    """
    
    print(f"Starting with {len(df)} records")
    
    # Step 1: Remove duplicates
    original_length = len(df)
    df = df.drop_duplicates()
    print(f"Removed {original_length - len(df)} duplicates")
    
    # Step 2: Handle missing values
    df["Sales"] = df["Sales"].fillna(df["Sales"].median())
    df["Region"] = df["Region"].fillna("Unknown")
    df = df.dropna(subset=["Date"])  # Date is critical
    
    # Step 3: Standardize strings
    df["Region"] = df["Region"].str.strip().str.title()
    df["Product"] = df["Product"].str.strip().str.title()
    
    # Step 4: Convert types
    df["Date"] = pd.to_datetime(df["Date"])
    df["Sales"] = pd.to_numeric(df["Sales"], errors="coerce")
    
    # Step 5: Remove outliers
    Q1 = df["Sales"].quantile(0.25)
    Q3 = df["Sales"].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df["Sales"] >= Q1 - 1.5*IQR) & 
            (df["Sales"] <= Q3 + 1.5*IQR)]
    
    print(f"Finished with {len(df)} clean records")
    
    return df

# Usage — re-create the original sample df to pass to the function
raw_df = pd.DataFrame({
    "Date": ["2024-01-15", "2024-01-16", None, "2024-01-18"],
    "Product": ["Laptop", "Mouse", "Laptop", "Keyboard"],
    "Sales": [1200, 25, 1200, None],
    "Region": ["North", "South", "North", "South"]
})
cleaned_df = clean_sales_data(raw_df)


Starting with 4 records
Removed 0 duplicates
Finished with 3 clean records


**Benefits of This Approach:**

- Consistency across multiple datasets
- Easy to modify cleaning rules
- Documented process (code explains what you did)
- Reusable for similar datasets

### Lab Session
**Lab 4: Comprehensive Data Cleaning Project**

**Objective:** Apply all data cleaning techniques to prepare a messy business dataset for analysis and visualization.

**Scenario:** You've been given a raw sales dataset exported from an old system. It has numerous quality issues that must be resolved before creating visualizations for the executive team. Your manager needs a clean dataset and a report documenting what was cleaned.

**Part A: Create Messy Dataset (5 points)**

Create file: `Lab04_YourName_DataCleaning.py`

Create this intentionally problematic dataset:


In [42]:
import pandas as pd
import numpy as np

data = {
    "Transaction_ID": [1001, 1002, 1003, 1004, 1005, 1006, 1002, 1007, 1008, 1009],
    "Date": ["2024-01-15", "2024-01-16", None, "2024-01-18", "2024-01-19", 
             "2024-01-20", "2024-01-16", "2024-01-21", "2024-01-22", "2024-01-23"],
    "Product": ["  Laptop", "MOUSE  ", "laptop", "Keyboard", "  Monitor", 
                "mouse", "MOUSE  ", "Tablet", "Headphones", "Laptop"],
    "Quantity": ["50", "200", "50", "150", "75", "200", "200", "100", "300", "2"],
    "Price": [1200, 25, 1200, 75, 350, 25, 25, 450, 80, 1200],
    "Sales": [60000, 5000, 60000, None, 26250, 5000, 5000, 45000, 24000, 2400],
    "Region": ["north", "SOUTH", "North", "south", None, "South", "SOUTH", 
               "East", "West", "North"],
    "Sales_Rep": ["Alice", "Bob", "Alice", "Charlie", "Diana", "Bob", "Bob", 
                  "Eve", "Frank", "Alice"]
}

df = pd.DataFrame(data)
print("Original messy data created")
print(df)


Original messy data created
   Transaction_ID        Date     Product Quantity  Price    Sales Region  \
0            1001  2024-01-15      Laptop       50   1200  60000.0  north   
1            1002  2024-01-16     MOUSE        200     25   5000.0  SOUTH   
2            1003        None      laptop       50   1200  60000.0  North   
3            1004  2024-01-18    Keyboard      150     75      NaN  south   
4            1005  2024-01-19     Monitor       75    350  26250.0   None   
5            1006  2024-01-20       mouse      200     25   5000.0  South   
6            1002  2024-01-16     MOUSE        200     25   5000.0  SOUTH   
7            1007  2024-01-21      Tablet      100    450  45000.0   East   
8            1008  2024-01-22  Headphones      300     80  24000.0   West   
9            1009  2024-01-23      Laptop        2   1200   2400.0  North   

  Sales_Rep  
0     Alice  
1       Bob  
2     Alice  
3   Charlie  
4     Diana  
5       Bob  
6       Bob  
7       Eve 

**Part B: Data Quality Assessment (15 points)**

Before cleaning, assess the problems:

1. **Check for missing values:**

   - Print count of missing values per column
   - Calculate percentage of missing data per column
   - Print rows that have any missing values

2. **Check for duplicates:**

   - Print duplicate rows (complete duplicates)
   - Check for duplicate Transaction_IDs
   - Count how many duplicates exist

3. **Check data types:**

   - Print current data type of each column
   - Identify which columns need type conversion

4. **Check for anomalies:**

   - Print basic statistics for numeric columns
   - Identify which values look suspicious

5. **Create a summary report:**

   Print a formatted report showing:
   - Total rows in original dataset
   - Number of missing values per column
   - Number of duplicate rows
   - Columns with wrong data types

**Part C: Step-by-Step Cleaning (50 points)**

Clean the data following these steps (print results after each step):

**Step 1: Handle Duplicates (10 points)**

- Identify and print duplicate rows
- Remove duplicate rows (keep first occurrence)
- Print how many duplicates were removed
- Print the DataFrame shape after removal

**Step 2: Handle Missing Values (10 points)**

- For "Sales" column: Fill missing values with the median sales
- For "Region" column: Fill missing values with "Unknown"
- For "Date" column: Fill missing values with "2024-01-17"
- Print the DataFrame to verify no missing values remain

**Step 3: Standardize Text Fields (10 points)**

- Product column: Remove whitespace, convert to title case
- Region column: Remove whitespace, capitalize properly
- Sales_Rep column: Ensure consistent capitalization
- Print unique values in Product and Region to verify

**Step 4: Convert Data Types (10 points)**

- Convert "Date" to datetime format
- Convert "Quantity" to integer type
- Convert "Price" to float type
- Verify "Sales" is float/numeric type
- Print data types to confirm changes

**Step 5: Validate and Correct Business Logic (10 points)**

- Recalculate Sales: Should equal Quantity × Price
- Create a new column "Sales_Calculated" with correct values
- Compare with existing "Sales" column
- Replace "Sales" with correct calculations
- Print any rows where Sales was incorrect

**Part D: Outlier Analysis (15 points)**

1. **Analyze Sales column for outliers:**

   - Calculate Q1, Q3, and IQR
   - Calculate lower and upper bounds
   - Print the bounds clearly
   - Identify and print outlier rows

2. **Business Decision:**

   - Review the outlier (Transaction_ID 1009)
   - This is a sale of 2 laptops at $1,200 each = $2,400
   - Determine if this is an error or legitimate
   - Document your decision with reasoning

3. **Create two versions:**

   - `df_with_outliers`: Keep all data
   - `df_no_outliers`: Remove outliers
   - Print record counts for both

**Part E: Create Cleaning Function (10 points)**

Write a reusable function called `clean_sales_dataset()` that:
1. Takes a raw DataFrame as input
2. Performs all cleaning steps from Parts C and D
3. Returns a clean DataFrame
4. Prints a summary of changes made

Test it by re-creating the messy dataset and running it through your function.

**Part F: Export and Documentation (5 points)**

1. Save the cleaned DataFrame to: `Lab04_YourName_CleanData.csv`
2. Create a text file: `Lab04_YourName_CleaningReport.txt` containing:

   - Original number of records
   - Number of duplicates removed
   - Number and location of missing values filled
   - Columns that were standardized
   - Data types that were converted
   - Number of outliers identified and decision made
   - Final number of clean records
   - Any assumptions or decisions you made

**Deliverables:**

1. `Lab04_YourName_DataCleaning.py` - Complete Python file with all parts
2. `Lab04_YourName_CleanData.csv` - Cleaned dataset
3. `Lab04_YourName_CleaningReport.txt` - Documentation of cleaning process

**Grading Rubric:**

- Data quality assessment: 15 points
- Cleaning steps completed correctly: 50 points
- Outlier analysis and justification: 15 points
- Reusable cleaning function: 10 points
- Documentation and export: 10 points

**Bonus Challenge (+10 points):**

Add advanced features to your cleaning function:

- Add parameter to control whether to remove outliers
- Add logging of each cleaning step to a file
- Create before/after comparison visualizations
- Handle edge cases (what if all values are duplicates?)

**Tips for Success:**

- Work through one step at a time
- Print intermediate results to verify each step
- Comment your code explaining WHY you made each decision
- Save your work frequently
- If stuck, clean the data manually first, then convert to function

**Common Pitfalls to Avoid:**

- Removing data without understanding why
- Using mean instead of median for skewed data
- Not verifying calculations (Sales = Quantity × Price)
- Over-cleaning (removing legitimate anomalies)
- Poor documentation (future you won't remember why you did something)

---

**End of Module 1: Foundations of Python and Data Handling**

**Key Takeaways:**

- Python and VSCode provide a powerful, professional environment for data work
- Python basics (variables, lists, loops, functions) enable automated data processing
- Pandas DataFrames are the core tool for business data analysis
- Real data is always messy - cleaning is essential before analysis
- Systematic cleaning workflows ensure consistent, reproducible results

**You're now ready to move to Module 2: Visualization!**
